# 09 · End-to-End Testing with Playwright

**Focus:** driving a real (headless) browser to test the app the way a user experiences it.

Unit and integration tests check your code in isolation. **End-to-end (E2E)** tests open an actual
browser, load a page, click things, and assert on what the user would see in the DOM. This catches
problems that only appear once HTML, CSS, and JavaScript run together.

**Playwright** launches a real Chromium browser and controls it from Python. The `pytest-playwright`
plugin provides a ready-to-use **`page`** fixture. We'll:
1. `%%writefile` a small static HTML page with a button that changes the DOM via JavaScript.
2. Load it in the browser, click the button, and assert the DOM updated.

Browsers run **headless** (no visible window) by default, which is exactly what you want in CI.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### A tiny web page
Clicking the button runs JS that flips the message text and reveals a hidden element.

In [2]:
%%writefile nb09_page.html
<!doctype html>
<html>
  <head><title>Playwright Demo</title></head>
  <body>
    <h1 id="status">Not clicked</h1>
    <button id="go" onclick="activate()">Click me</button>
    <p id="secret" style="display:none">You found me!</p>
    <script>
      function activate() {
        document.getElementById('status').textContent = 'Clicked!';
        document.getElementById('secret').style.display = 'block';
      }
    </script>
  </body>
</html>

Writing nb09_page.html


### The E2E test
`page` is injected by `pytest-playwright`. We navigate to the file with a `file://` URL, assert the
initial state, click `#go`, then assert the DOM changed. `expect(...)` auto-waits for the condition,
so there are no flaky `sleep`s.

In [3]:
%%writefile test_nb09.py
import os
from playwright.sync_api import Page, expect

# Absolute file:// URL to the HTML we wrote next to this test.
PAGE_URL = "file://" + os.path.abspath("nb09_page.html")

def test_button_updates_dom(page: Page):
    page.goto(PAGE_URL)

    # Initial state
    expect(page.locator("#status")).to_have_text("Not clicked")
    expect(page.locator("#secret")).to_be_hidden()

    # Interact
    page.click("#go")

    # Asserted post-click state (expect() auto-waits for the JS to run)
    expect(page.locator("#status")).to_have_text("Clicked!")
    expect(page.locator("#secret")).to_be_visible()
    expect(page.locator("#secret")).to_have_text("You found me!")

Writing test_nb09.py


### Run it
`pytest-playwright` defaults to headless Chromium. (The browser binary was installed once via
`playwright install chromium`.) `--browser chromium` is explicit here for clarity.

In [4]:
!pytest -v --browser chromium test_nb09.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item                                                               

test_nb09.py::test_button_updates_dom[chromium] 

PASSED                   [100%]



============================== 1 passed in 0.84s ===============================


### ⚠️ Common pitfall — never `sleep()`, let `expect()` wait
The classic E2E flake is `time.sleep(2)` after an action, hoping the DOM caught up. Playwright's
`expect(locator).to_...()` **auto-retries** until the condition holds or a timeout expires — so it's
both faster (proceeds the instant it's ready) and reliable (no guessing). If an assertion is wrong, it
waits out the full timeout and then reports a clear diff.

### 🔬 Try it yourself
This test clicks the button but then asserts the **wrong** text, with a short 2-second timeout.
**Predict:** how long until it fails, and what does the message say? Run it — notice `expect` waited
(auto-retrying) and then reported the actual vs expected text.

In [5]:
%%writefile test_nb09_tryit.py
import os
from playwright.sync_api import Page, expect

PAGE_URL = "file://" + os.path.abspath("nb09_page.html")

def test_expect_autowaits_then_fails(page: Page):
    page.goto(PAGE_URL)
    page.click("#go")
    # WRONG on purpose: the real text becomes "Clicked!". Short timeout so it fails fast.
    expect(page.locator("#status")).to_have_text("Not clicked yet", timeout=2000)

Writing test_nb09_tryit.py


In [6]:
!pytest -v --browser chromium test_nb09_tryit.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 1 item                                                               

test_nb09_tryit.py::test_expect_autowaits_then_fails[chromium] 

FAILED    [100%]

=================================== FAILURES ===================================
__________________ test_expect_autowaits_then_fails[chromium] __________________

page = <Page url='file:///Users/ajinkyak/Desktop/JULY-WORK/a%20-%20training_assignments/explorable_codelesson/nb09_page.html'>



    def test_expect_autowaits_then_fails(page: Page):
        page.goto(PAGE_URL)
        page.click("#go")
        # WRONG on purpose: the real text becomes "Clicked!". Short timeout so it fails fast.
>       expect(page.locator("#status")).to_have_text("Not clicked yet", timeout=2000)
E       AssertionError: Locator expected to have text 'Not clicked yet'
E       Actual value: Clicked! 
E       Call log:
E         - Expect "to_have_text" with timeout 2000ms
E         - waiting for locator("#status")
E           21 × locator resolved to <h1 id="status">Clicked!</h1>
E              - unexpected value "Clicked!"
E       
E       Aria snapshot:
E       - heading "Clicked!" [level=1]

test_nb09_tryit.py:10: AssertionError
=========================== short test summary info ============================
FAILED test_nb09_tryit.py::test_expect_autowaits_then_fails[chromium] - AssertionError: Locator expected to have text 'Not clicked yet'
============================== 1 failed in 2.76s =====

### What you learned
- Playwright drives a real browser; `pytest-playwright` gives you the `page` fixture for free.
- `page.goto(url)`, `page.click(selector)` interact; `expect(locator).to_have_text/to_be_visible(...)`
  assert — and **auto-wait**, so tests aren't flaky.
- Tests run **headless** by default — ideal for CI.
- You can point Playwright at a live URL or, as here, a local `file://` page.

Next up: the finale — **coverage vs. mutation testing**.